Hello Deena, welcome to my notebook showcasing my automl library. If you are interested you can look at the diagram in the chat.

The goal of the program is that the user can easily use advanced predifined or custom hypertuning and feature selection methods on their dataset.

The user needs to supply the target column as a string, the dataset as a df, as well as either choose from the predifined hypertuning and feature selection methods or make their own which uphold a interface.

The run_automl method first runs the feature selection function if defined. then the hypertuning method if defined.



In [ ]:

import sys
import pandas as pd
from pathlib import Path

#the import process will be different if made like a true library
code = Path.cwd()/'code'
sys.path.append(str(code))
from automl.automl import SimpleAutoML
from Loss import mae
from feature_selection.forward import ForwardFeatureSelector
from feature_selection.backwards import BackwardFeatureSelector  
from hyper_tuning import GridSearchTuner, RandomSearchTuner, LineSearchTuner  


In [ ]:

data_path='cleaned_data_harsh.csv'

df=pd.read_csv(data_path)
df=df.sample(frac=0.05, random_state=42) #5 percent of the data

df_clean=df.drop(columns=['dato'])
print(f"data shape {df_clean.shape}")

automl=SimpleAutoML()


Cleaned data shape: (76117, 21)
Failed to load model from /Users/victor/School/bachelor 2/bachelor/automltrainer/code/models/lightgbm.py: No module named 'lightgbm'
Registered model: linear_regression
Registered model: xgboost


In [ ]:
# a simple run through with no feature selection and no hyperparameter tuning

results = automl.run_automl( df=df_clean,
    target_col='Købesum', # target column
    feature_selection_fn=None, # how you specify no feature selection
    models_to_run=['linear_regression', 'xgboost'], # models we would like to include if we dont supply this then all valid models are chosen
    hypertuning_fn=None, # how you specify no hyperparameter tuning
    loss_fn=mae(), # loss function to use
    n_splits=5, # number of splits for cross-validation i use scikitlearn time series split
    test_split=0.1) # percentage of the data to use for testing

Failed to load model from /Users/victor/School/bachelor 2/bachelor/automltrainer/code/models/lightgbm.py: No module named 'lightgbm'
Registered model: linear_regression
Registered model: xgboost
Starting AutoML Pipeline - Training ALL available models...
Data split - Train: 68505, Test: 7612
Training 2 models: ['linear_regression', 'xgboost']

Training linear_regression
  Using default parameters for linear_regression
✓ linear_regression - Test mae: 1196648.85 (Features: 20)

Training xgboost
  Using default parameters for xgboost
✓ xgboost - Test mae: 916447.56 (Features: 20)

AUTOML RESULTS
linear_regression: Test mae: 1,196,648.85
xgboost: Test mae: 916,447.56

Best Model: xgboost
Best Test mae: 916,447.56


In [ ]:

save_path = automl.save_model('saved_models/Deenaexample1') # how we can save the model and specification like the scalar and columns we used

Model package saved to: saved_models/Deenaexample1.pkl
Model weights saved to: saved_models/Deenaexample1_xgboost.json


In [ ]:
# example with feature selection
results = automl.run_automl( df=df_clean,
    target_col='Købesum',
    feature_selection_fn=ForwardFeatureSelector,
    models_to_run=[ 'xgboost','linear_regression',],
    hypertuning_fn=None,
    loss_fn=mae(),
    n_splits=5,
    test_split=0.1)

Starting AutoML Pipeline - Training ALL available models...
Data split - Train: 68505, Test: 7612
Training 2 models: ['xgboost', 'linear_regression']

Training xgboost
  Running feature selection for xgboost
Starting forward feature selection with 20 available features
Added feature: mean_of_5_neighbors_pris, CV Score: 1075762.0250, Selected: 1
Added feature: pris_pr_m2_mean_365D_by, CV Score: 1027584.5375, Selected: 2
Added feature: age, CV Score: 999343.1250, Selected: 3
Added feature: m2, CV Score: 967741.4000, Selected: 4
Added feature: std_of_5_neighbors_pris_pr_m2, CV Score: 954004.0000, Selected: 5
Added feature: btype_Villa, CV Score: 942266.4750, Selected: 6
Added feature: btype_Fritidshus, CV Score: 935464.4250, Selected: 7
Added feature: omr_de_Hovedstaden__K_benhavn, CV Score: 930185.6000, Selected: 8
Added feature: date_ordinal, CV Score: 923547.8750, Selected: 9
Added feature: btype_R_kkehus, CV Score: 921699.9375, Selected: 10
Added feature: omr_de_Nordsj_lland, CV Score

In [ ]:
# another type of feature selection, cool huh? 

results = automl.run_automl( df=df_clean,
    target_col='Købesum',
    feature_selection_fn=BackwardFeatureSelector,
    models_to_run=[ 'xgboost','linear_regression',],
    hypertuning_fn=None,
    loss_fn=mae(),
    n_splits=5,
    test_split=0.1)

Starting AutoML Pipeline - Training ALL available models...
Data split - Train: 68505, Test: 7612
Training 2 models: ['xgboost', 'linear_regression']

Training xgboost
  Running feature selection for xgboost
Starting backward feature selection with 20 features
Initial CV Score: 919094.4500
Dropped feature: btype_Landejendom, CV Score: 917700.2875, Remaining: 19
Dropped feature: omr_de_Bornholm, CV Score: 917190.6250, Remaining: 18
No improvement found (best would be 917587.9500 vs current 917190.6250), stopping feature selection
Feature selection complete. Selected 18 features
Final CV Score: 917190.6250
  Features after selection for xgboost: 18
  Using default parameters for xgboost
✓ xgboost - Test mae: 919732.12 (Features: 18)

Training linear_regression
  Running feature selection for linear_regression
Starting backward feature selection with 20 features
Initial CV Score: 1188334.7955
Dropped feature: btype_Villa, CV Score: 1184978.0877, Remaining: 19
Dropped feature: age, CV Scor

In [27]:
# hypertuning

results = automl.run_automl( df=df_clean,
    target_col='Købesum',
    feature_selection_fn=None,
    models_to_run=[ 'xgboost','linear_regression',],
    hypertuning_fn=RandomSearchTuner,
    loss_fn=mae(),
    n_splits=5,
    test_split=0.1)

Starting AutoML Pipeline - Training ALL available models...
Data split - Train: 68505, Test: 7612
Training 2 models: ['xgboost', 'linear_regression']

Training xgboost
  Running hyperparameter tuning for xgboost
Testing 10 random parameter combinations


Exception ignored on calling ctypes callback function <bound method DataIter._next_wrapper of <xgboost.data.SingleBatchInternalIter object at 0x311c70ec0>>:
Traceback (most recent call last):
  File "/Users/victor/anaconda3/envs/bachelor/lib/python3.13/site-packages/xgboost/core.py", line 561, in _next_wrapper
    def _next_wrapper(self, this: None) -> int:  # pylint: disable=unused-argument
KeyboardInterrupt: 


✗ xgboost failed: [15:23:48] /Users/runner/miniforge3/conda-bld/xgboost-split_1748292887431/work/src/data/quantile_dmatrix.cc:174: Check failed: accumulated_rows == info.num_row_ (57088 vs. 114176) : 
Stack trace:
  [bt] (0) 1   libxgboost.dylib                    0x0000000171cbe0f0 dmlc::LogMessageFatal::~LogMessageFatal() + 124
  [bt] (1) 2   libxgboost.dylib                    0x0000000171e74d6c xgboost::data::cpu_impl::MakeSketches(xgboost::Context const*, xgboost::data::DataIterProxy<void (void*), int (void*)>*, xgboost::data::DMatrixProxy*, std::__1::shared_ptr<xgboost::DMatrix>, float, xgboost::common::HistogramCuts*, xgboost::BatchParam const&, xgboost::MetaInfo const&, xgboost::data::ExternalDataInfo const&, std::__1::vector<xgboost::FeatureType, std::__1::allocator<xgboost::FeatureType>>*) + 2736
  [bt] (2) 3   libxgboost.dylib                    0x0000000171e64d00 xgboost::data::IterativeDMatrix::InitFromCPU(xgboost::Context const*, xgboost::BatchParam const&, void*, float, 

In [28]:
#not really made in a cool way, should be refactored, but it is possible to alter the parameters of the predifined functions.

def tuner( estimator, loss_fn, param_grid, cv, n_jobs, verbose,):
    return RandomSearchTuner(
        estimator=estimator,
        loss_fn=loss_fn,
        param_grid=param_grid, 
        cv=cv,
        n_iter=25,
        verbose=verbose # here we can directly change the number of iterations for the random search tuning
    )

results = automl.run_automl( df=df_clean,
    target_col='Købesum',
    feature_selection_fn=None,
    models_to_run=[ 'xgboost','linear_regression',],
    hypertuning_fn=tuner,
    loss_fn=mae(),
    n_splits=5,
    test_split=0.1)

Starting AutoML Pipeline - Training ALL available models...
Data split - Train: 68505, Test: 7612
Training 2 models: ['xgboost', 'linear_regression']

Training xgboost
  Running hyperparameter tuning for xgboost
Testing 25 random parameter combinations


KeyboardInterrupt: 

In [ ]:
# two more examples with hypertuning
results = automl.run_automl( df=df_clean,
    target_col='Købesum',
    feature_selection_fn=None,
    models_to_run=[ 'xgboost','linear_regression',],
    hypertuning_fn=GridSearchTuner,
    loss_fn=mae(),
    n_splits=5,
    test_split=0.1)

In [ ]:
#via the parameter param_amount we can specifiy how many parameter space n we want to have
results = automl.run_automl( df=df_clean,
    target_col='Købesum',
    feature_selection_fn=None,
    models_to_run=[ 'xgboost','linear_regression',],
    hypertuning_fn=LineSearchTuner,
    loss_fn=mae(),
    n_splits=5,
    test_split=0.1,
    param_amount='big')

Starting AutoML Pipeline - Training ALL available models...
Data split - Train: 68505, Test: 7612
Training 2 models: ['xgboost', 'linear_regression']

Training xgboost
  Running hyperparameter tuning for xgboost
Starting Line Search with initial params: {'n_estimators': 100, 'learning_rate': 0.01, 'max_depth': 3, 'subsample': 0.8, 'colsample_bytree': 0.8}

--- Pass 1/2 ---
  New best for 'n_estimators': 750 -> Score: 960064.3000
  New best for 'learning_rate': 0.1 -> Score: 918657.4625
  New best for 'colsample_bytree': 1.0 -> Score: 917953.4500

--- Pass 2/2 ---
Stopping early, no improvement in a full pass.

Best parameters found: {'n_estimators': 750, 'learning_rate': 0.1, 'max_depth': 3, 'subsample': 0.8, 'colsample_bytree': 1.0}
Best CV score: 917953.4500
  Best params for xgboost: {'n_estimators': 750, 'learning_rate': 0.1, 'max_depth': 3, 'subsample': 0.8, 'colsample_bytree': 1.0}
✓ xgboost - Test mae: 922632.31 (Features: 20)

Training linear_regression
  Running hyperparameter

KeyboardInterrupt: 